# COMP47470 Assignment 3: Apache Spark DataFrames and Spark SQL

This assignment is about DataFrame operations (Transformations and Actions) and Spark SQL on structured data in Apache Spark.

**I have used the Google Colab to complete this assignment**

The assignment carries 10 points (10% weight). It has two components. The first component is on DataFrame transformations. The second component is on Spark SQL.

## Component 1: Apache Spark DataFrame Operations

For the questions in this component, you will use the **Air Quality Health Risk Assessments** dataset.

https://www.eea.europa.eu/en/datahub/datahubitem-view/49930245-dc33-4c47-93b8-9512f0622ebc


The data fields are as follows (given by the header line):

```python
City Boundary Specification (LAU/grid)
Country Or Territory
City
City Code
Total City Population *
Year
Air Pollutant
Health Risk Scenario
Populated Area [km2]
Air Pollution Average [ug/m3]
Air Pollution Population Weighted Average [ug/m3]
Premature Deaths
Premature Deaths - lower CI
Premature Deaths - upper CI
Years Of Life Lost
Years Of Life Lost - lower CI
Years Of Life Lost - upper CI
```

**Premature death** refers to a situation where part of a lifespan is lost due to an event that shortens life expectancy, resulting in a few years of life being lost that would not normally occur in a natural death.

**Years of life lost** (YLL) is defined as the years of potential life lost because of premature death. YLL is an estimate of the number of years that people in a population would have lived had there been no premature deaths.

In [1]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://dlcdn.apache.org/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

import os
import sys
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.2.1-bin-hadoop3.2"


import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark= SparkSession \
       .builder \
       .appName("Assignment 3") \
       .getOrCreate()

spark

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:6 https://cli.github.com/packages stable InRelease
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,856 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [

In [2]:
spark

### Problem 1: What are the Spark operations that create a DataFrame using the input CSV file **Air_Quality_Health_Risk_Assessments.csv** and prints the schema?

The Spark read option must infer the schema from the CSV file to create a DataFrame with name **airquality_df**.

The Spark print schema method on the DataFrame must print the human readable schema.

**What are the Spark DataFrame Operations and output from their execution?**

In [3]:
from google.colab import files
uploaded = files.upload()

Saving Air_Quality_Health_Risk_Assessments.csv to Air_Quality_Health_Risk_Assessments.csv
Saving CO2_vans_v1.csv to CO2_vans_v1.csv


In [4]:
csv_file_name = "Air_Quality_Health_Risk_Assessments.csv"

airquality_df = spark.read.csv('/content/' + csv_file_name, header=True, inferSchema=True)

airquality_df.printSchema()

airquality_df.show(5)

root
 |-- City Boundary Specification (LAU/grid): string (nullable = true)
 |-- Country Or Territory: string (nullable = true)
 |-- City: string (nullable = true)
 |-- City Code: string (nullable = true)
 |-- Total City Population *: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Air Pollutant: string (nullable = true)
 |-- Health Risk Scenario: string (nullable = true)
 |-- Populated Area [km2]: integer (nullable = true)
 |-- Air Pollution Average [ug/m3]: double (nullable = true)
 |-- Air Pollution Population Weighted Average [ug/m3]: double (nullable = true)
 |-- Premature Deaths: integer (nullable = true)
 |-- Premature Deaths - lower CI: integer (nullable = true)
 |-- Premature Deaths - upper CI: integer (nullable = true)
 |-- Years Of Life Lost: integer (nullable = true)
 |-- Years Of Life Lost - lower CI: integer (nullable = true)
 |-- Years Of Life Lost - upper CI: integer (nullable = true)

+--------------------------------------+--------------------+-----

### Problem 2: What are the Spark DataFrame operations that determine the **total** number of premature deaths and years of life lost for the City of Dublin?

The city is given by the column **City**.

The number of premature deaths is given by the column **Premature Deaths**.

The number of years of life lost is given by the column **Years Of Life Lost**.

**What are the Spark DataFrame Operations and output from their execution?**

In [7]:
dublin_totals = (
    airquality_df
        .filter(F.col("City") == "Dublin")
        .groupBy("City")
        .agg(
            F.sum("Premature Deaths"),
            F.sum("Years Of Life Lost")
        )
)

dublin_totals.show()

+------+---------------------+-----------------------+
|  City|sum(Premature Deaths)|sum(Years Of Life Lost)|
+------+---------------------+-----------------------+
|Dublin|                 5841|                  70246|
+------+---------------------+-----------------------+



### Problem 3: What are the Spark DataFrame operations that determine the air pollutant that caused the maximum **total** number of premature deaths for the City of Dublin?

The city is given by the column **City**.

The number of premature deaths is given by the column **Premature Deaths**.

The output must contain the following:

1.   Air pollutant
2.   Total number of premature deaths caused by this pollutant

**What are the Spark DataFrame Operations and output from their execution?**

In [8]:
max_pollutant = airquality_df.filter(F.col("City") == "Dublin") \
    .groupBy("Air Pollutant") \
    .agg(F.sum("Premature Deaths").alias("Total Premature Deaths")) \
    .orderBy(F.desc("Total Premature Deaths")) \
    .limit(1)

max_pollutant.show()

+-------------+----------------------+
|Air Pollutant|Total Premature Deaths|
+-------------+----------------------+
|        PM2.5|                  3826|
+-------------+----------------------+



### Problem 4: What are the Spark DataFrame operations that determine the country with the maximum **total** number of premature deaths?

The country is given by the column **Country Or Territory**.

Note the country must **not** be any of the following:

```Python
All Countries
European Union Countries
European Environment Agency Member Countries
```

The output must contain the following:

1.   Country
2.   Total number of premature deaths for the country

**What are the Spark DataFrame Operations and output from their execution?**

In [9]:
max_country = airquality_df.filter(
    ~F.col("Country Or Territory").isin(
        ["All Countries", "European Union Countries", "European Environment Agency Member Countries"]
    )
).groupBy("Country Or Territory") \
 .agg(F.sum("Premature Deaths").alias("Total Premature Deaths")) \
 .orderBy(F.desc("Total Premature Deaths")) \
 .limit(1)

max_country.show()

+--------------------+----------------------+
|Country Or Territory|Total Premature Deaths|
+--------------------+----------------------+
|               Italy|                886838|
+--------------------+----------------------+



## Component 2: Apache Spark SQL

For the questions in this component, you will use the **Monitoring of CO2 emissions from vans - Data 2014** dataset.

The data fields are tabulated in the webpage below:

https://www.eea.europa.eu/data-and-maps/data/vans-18/monitoring-of-co2-emissions-from


The data fields are as follows:

```python
Field Name Field Definition 			Field Type
---------- ---------------- 			----------
id         ID               			Integer      
MS         Member state     			String
MP         Manufacturer pooling 		String
Mh         Manufacturer harmonised 		String
MAN        Manufacturer name OEM		String
MMS        Manufacturer name as in MS		String
TAN        Type approval number 		String
T          Type                 		String
Va         Variant              		String
Ve         Version              		String      
Mk         Make                 		String      
Cn         Commerical Name      		String      
Ct         Category of the vehicle type   	String      
Cr         Category of the vehicle registered	String
r          Total new registrations 		Integer
m (kg)     Mass                    		Integer
TPMLM (kg) Maximum laden mass      		Integer     
e (g/km)   Specific CO2 Emissions  		Integer
w (mm)     Wheel Base              		Integer
at1 (mm)   Axle width steering axle 		Integer
at2 (mm)   Axle width other axle    		Integer
Ft         Fuel Type                		String
Fm         Fuel Mode                		String
ec (cm3)   Engine Capacity          		Integer     
ep (KW)    Engine Power             		Integer
z (Wh/km)  Electric energy consumption 		Integer
IT         Innovative technology       		String
Er (g/km)  Emissions reduction through IT 	Integer
```

### Problem 1: Write the schema using the Spark DataFrame StructType interface and load the dataset into a Spark DataFrame using the schema.

For example:

```python
co2emissionsschema = StructType([StructField('id', IntegerType(), True),
                                 StructField('MS', StringType(), True),
                                 StructField('MP', StringType(), True),
...
```

Load the dataset using the Spark DataFrame load operation with the filename and the schema as inputs.

**What are the schema and the Spark DataFrame Operations?** 

In [10]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType


In [11]:
co2emissionsschema = StructType([
    StructField('id', IntegerType(), True),
    StructField('MS', StringType(), True),
    StructField('MP', StringType(), True),
    StructField('Mh', StringType(), True),
    StructField('MAN', StringType(), True),
    StructField('MMS', StringType(), True),
    StructField('TAN', StringType(), True),
    StructField('T', StringType(), True),
    StructField('Va', StringType(), True),
    StructField('Ve', StringType(), True),
    StructField('Mk', StringType(), True),
    StructField('Cn', StringType(), True),
    StructField('Ct', StringType(), True),
    StructField('Cr', StringType(), True),
    StructField('r', IntegerType(), True),
    StructField('m (kg)', IntegerType(), True),
    StructField('TPMLM (kg)', IntegerType(), True),
    StructField('e (g/km)', IntegerType(), True),
    StructField('w (mm)', IntegerType(), True),
    StructField('at1 (mm)', IntegerType(), True),
    StructField('at2 (mm)', IntegerType(), True),
    StructField('Ft', StringType(), True),
    StructField('Fm', StringType(), True),
    StructField('ec (cm3)', IntegerType(), True),
    StructField('ep (KW)', IntegerType(), True),
    StructField('z (Wh/km)', IntegerType(), True),
    StructField('IT', StringType(), True),
    StructField('Er (g/km)', IntegerType(), True)
])


In [12]:
co2_df = spark.read.csv(
    '/content/CO2_vans_v1.csv',
    sep='\t',
    header=True,
    schema=co2emissionsschema
)

co2_df.printSchema()

co2_df.show(5)


root
 |-- id: integer (nullable = true)
 |-- MS: string (nullable = true)
 |-- MP: string (nullable = true)
 |-- Mh: string (nullable = true)
 |-- MAN: string (nullable = true)
 |-- MMS: string (nullable = true)
 |-- TAN: string (nullable = true)
 |-- T: string (nullable = true)
 |-- Va: string (nullable = true)
 |-- Ve: string (nullable = true)
 |-- Mk: string (nullable = true)
 |-- Cn: string (nullable = true)
 |-- Ct: string (nullable = true)
 |-- Cr: string (nullable = true)
 |-- r: integer (nullable = true)
 |-- m (kg): integer (nullable = true)
 |-- TPMLM (kg): integer (nullable = true)
 |-- e (g/km): integer (nullable = true)
 |-- w (mm): integer (nullable = true)
 |-- at1 (mm): integer (nullable = true)
 |-- at2 (mm): integer (nullable = true)
 |-- Ft: string (nullable = true)
 |-- Fm: string (nullable = true)
 |-- ec (cm3): integer (nullable = true)
 |-- ep (KW): integer (nullable = true)
 |-- z (Wh/km): integer (nullable = true)
 |-- IT: string (nullable = true)
 |-- Er (g/km

### Problem 2: Write a Spark SQL query that outputs the van details with the maximum specific CO2 emissions given by e (g/km)?

This query will find the data record with the maximum e (g/km) value in the dataset.

Create a temporary view (table) called **co2emissionstbl** from the DataFrame created in the previous question.

Note you can select a column with spaces in its name using backticks. For example, to select the column **e (g/km)** in a query, use \`e (g/km)\`.

The output of the query must contain the following:

id, MMS, Mk, Cn, e (g/km)

**What is the Spark SQL query and the output from the query execution?**

In [13]:
# Create a temporary SQL view
co2_df.createOrReplaceTempView("co2emissionstbl")

spark.sql("""
SELECT id, MMS, Mk, Cn, `e (g/km)`
FROM co2emissionstbl
WHERE `e (g/km)` = (SELECT MAX(`e (g/km)`) FROM co2emissionstbl)
""").show()

+-----+----------+----------+----+--------+
|   id|       MMS|        Mk|  Cn|e (g/km)|
+-----+----------+----------+----+--------+
|60905|VOLKSWAGEN|VOLKSWAGEN|NULL|     490|
+-----+----------+----------+----+--------+



### Problem 3: Write the Spark SQL query that outputs the van details with the maximum **total** specific CO2 emissions given by e (g/km)?

The query will find the van (grouped by **Mk** and **Ft**) that has contributed the maximum total CO2 emissions.

You will capture the total specific CO2 emissions in a field called **TotalEmissions**.

The output of the query must contain the following fields:

**Mk, Ft, TotalEmissions**

**What is the Spark SQL query and the output from the query execution?**

In [14]:
spark.sql("""
SELECT Mk, Ft, SUM(`e (g/km)`) AS TotalEmissions
FROM co2emissionstbl
GROUP BY Mk, Ft
ORDER BY TotalEmissions DESC
LIMIT 1
""").show()


+-------------+------+--------------+
|           Mk|    Ft|TotalEmissions|
+-------------+------+--------------+
|MERCEDES-BENZ|Diesel|       1976224|
+-------------+------+--------------+



In [15]:
spark.stop()